# MedForge Free TotalSegmentator Worker

This notebook is a free/low-cost segmentation worker for **MedForge Imaging Studio**. It runs TotalSegmentator on a de-identified DICOM ZIP or NIfTI volume, then downloads a ZIP of NIfTI masks that MedForge can import.

**Important:** Google Colab is an external cloud service. Use de-identified imaging unless you have independently decided it is appropriate to upload identifiable health information. For private/local processing, run TotalSegmentator or 3D Slicer on your own computer instead.

TotalSegmentator is not a medical device. Its outputs require review before being used as anatomical evidence.

In [ ]:
!pip -q install TotalSegmentator nibabel
import os, shutil, subprocess, sys
from pathlib import Path
try:
    import torch
    print('GPU available:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as e:
    print('Torch check:', e)


## 1. Upload one de-identified DICOM ZIP or NIfTI file
For CT use task `total`; for MR use `total_mr`. The default tasks are the sensible starting point for MedForge.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one DICOM ZIP or NIfTI file for this worker.')
input_name, input_bytes = next(iter(uploaded.items()))
input_path = Path('/content') / input_name
input_path.write_bytes(input_bytes)
print('Saved:', input_path)


## 2. Choose CT or MR
Free Colab GPUs are not guaranteed. If there is no GPU, this notebook automatically uses TotalSegmentator's faster/lower-resolution mode.

In [ ]:
# Change this to 'MR' for an MR study.
MODALITY = 'CT'  # 'CT' or 'MR'
TASK = 'total_mr' if MODALITY.upper() == 'MR' else 'total'
OUTPUT_DIR = Path('/content/medforge_segmentations')
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir()

device_args = []
fast_args = []
try:
    import torch
    if torch.cuda.is_available():
        device_args = ['--device', 'gpu']
    else:
        fast_args = ['--fast']
except Exception:
    fast_args = ['--fast']

cmd = ['TotalSegmentator', '-i', str(input_path), '-o', str(OUTPUT_DIR), '--task', TASK] + device_args + fast_args
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Masks created:', len(list(OUTPUT_DIR.glob('*.nii.gz'))))


## 3. Package masks for MedForge
MedForge treats these as **DERIVED segmentation data**, not original imaging and not a diagnosis.

In [ ]:
manifest = OUTPUT_DIR / 'MEDFORGE_SEGMENTATION_NOTE.txt'
manifest.write_text(
    'Generated by TotalSegmentator for MedForge.\\n'
    f'Task: {TASK}\\n'
    'Status: DERIVED / UNREVIEWED segmentation.\\n'
    'Do not treat a generated mask as a diagnosis or as source imaging.\\n'
)
archive = shutil.make_archive('/content/medforge_totalsegmentator_masks', 'zip', OUTPUT_DIR)
print('Created:', archive)
files.download(archive)


## 4. Back in MedForge
Open **MPR & 3D → Import reviewed/generated anatomy masks**, upload the downloaded ZIP, inspect the mask planes and 3D surface, and keep the mask labeled as derived/unreviewed until a human reviews it.